In [ ]:
%%capture
!pip install unsloth

In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'

In [ ]:
import re
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

from unsloth import FastLanguageModel
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import set_seed
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_all_seeds(42)

In [ ]:
N_SAMPLES = 5000
KEEP_FIELDS = {"url", "label", "lang"}

stream = load_dataset("phreshphish/phreshphish", split="train", streaming=True)
sampled = Dataset.from_list(
    [{k: row[k] for k in KEEP_FIELDS} for row in stream.take(N_SAMPLES)]
)
print(f"Sampled {len(sampled)} examples")
print(sampled[0])

In [ ]:
print(f"=== Raw sampled dataset ===")
print(f"Total samples: {len(sampled)}")
print(f"Columns: {sampled.column_names}")
lang_counts = pd.Series(sampled['lang']).value_counts()
english_count = lang_counts.get('en', 0)
print(f"English: {english_count} ({english_count/len(sampled)*100:.1f}%) | Other languages: {len(sampled) - english_count}")
print(f"Labels: {pd.Series(sampled['label']).value_counts().to_string()}")
print()

dataset = sampled.filter(lambda x: x["lang"] == "en")
non_english_count = len(sampled) - len(dataset)
print(f"=== After language filter ===")
print(f"Kept: {len(dataset)} English | Removed: {non_english_count} non-English ({non_english_count/len(sampled)*100:.1f}%)")
print(f"Labels: {pd.Series(dataset['label']).value_counts().to_string()}")

In [ ]:
from urllib.parse import urlparse
from collections import Counter

all_urls = dataset["url"]
all_labels = dataset["label"]
phish_urls = [u for u, l in zip(all_urls, all_labels) if l == "phish"]
benign_urls = [u for u, l in zip(all_urls, all_labels) if l == "benign"]

def url_features(urls):
    tlds, url_lens, path_depths = [], [], []
    has_hyphen, has_ip, has_at = 0, 0, 0
    for url in urls:
        p = urlparse(url)
        domain = p.netloc
        parts = domain.split(".")
        if len(parts) >= 2:
            tlds.append(parts[-1])
        url_lens.append(len(url))
        path_depths.append(len([s for s in p.path.split("/") if s]))
        has_hyphen += "-" in domain
        has_ip += bool(re.match(r"\d+\.\d+\.\d+\.\d+", domain))
        has_at += "@" in url
    return tlds, url_lens, path_depths, has_hyphen, has_ip, has_at

p_tlds, p_lens, p_depths, p_hyp, p_ip, p_at = url_features(phish_urls)
b_tlds, b_lens, b_depths, b_hyp, b_ip, b_at = url_features(benign_urls)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

top_n = 10
p_tld_counts = Counter(p_tlds)
b_tld_counts = Counter(b_tlds)
all_top_tlds = [t for t, _ in (p_tld_counts + b_tld_counts).most_common(top_n)]
x = np.arange(len(all_top_tlds))
w = 0.35
axes[0].bar(x - w/2, [100 * p_tld_counts[t] / len(phish_urls) for t in all_top_tlds], w, label="Phish", color="crimson", alpha=0.8)
axes[0].bar(x + w/2, [100 * b_tld_counts[t] / len(benign_urls) for t in all_top_tlds], w, label="Benign", color="steelblue", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f".{t}" for t in all_top_tlds], rotation=45)
axes[0].set_ylabel("% of class")
axes[0].set_title("TLD Distribution")
axes[0].legend()

all_lens = p_lens + b_lens
lo, hi = np.percentile(all_lens, 1), np.percentile(all_lens, 99)
p_lens_clip = [l for l in p_lens if lo <= l <= hi]
b_lens_clip = [l for l in b_lens if lo <= l <= hi]
shared_bins = np.linspace(lo, hi, 40)
axes[1].hist(p_lens_clip, bins=shared_bins, alpha=0.6, label="Phish", color="crimson", density=True)
axes[1].hist(b_lens_clip, bins=shared_bins, alpha=0.6, label="Benign", color="steelblue", density=True)
axes[1].set_xlabel("URL length (chars)")
axes[1].set_ylabel("Density")
axes[1].set_title("URL Length Distribution")
axes[1].legend()

max_depth = 8
p_depth_counts = [sum(1 for d in p_depths if d == i) / len(phish_urls) * 100 for i in range(max_depth + 1)]
b_depth_counts = [sum(1 for d in b_depths if d == i) / len(benign_urls) * 100 for i in range(max_depth + 1)]
x4 = np.arange(max_depth + 1)
axes[2].bar(x4 - w/2, p_depth_counts, w, label="Phish", color="crimson", alpha=0.8)
axes[2].bar(x4 + w/2, b_depth_counts, w, label="Benign", color="steelblue", alpha=0.8)
axes[2].set_xlabel("Path depth (# of segments)")
axes[2].set_ylabel("% of class")
axes[2].set_title("URL Path Depth")
axes[2].legend()

plt.suptitle("Phishing vs Benign: URL Signal Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Phish: hyphen_in_domain={100*p_hyp/len(phish_urls):.1f}%, avg_url_len={np.mean(p_lens):.1f}, avg_path_depth={np.mean(p_depths):.1f}")
print(f"Benign: hyphen_in_domain={100*b_hyp/len(benign_urls):.1f}%, avg_url_len={np.mean(b_lens):.1f}, avg_path_depth={np.mean(b_depths):.1f}")

In [ ]:
dataset = dataset.remove_columns(["lang"])
print(f"Remaining columns: {dataset.column_names}")
print(f"Total samples: {len(dataset)}")

In [ ]:
benign = dataset.filter(lambda x: x["label"] == "benign")
phish = dataset.filter(lambda x: x["label"] == "phish")
min_count = min(len(benign), len(phish))

print(f"=== Balancing ===")
print(f"Before: benign={len(benign)}, phish={len(phish)}")
print(f"Downsampling to {min_count} per class")

benign = benign.shuffle(seed=42).select(range(min_count))
phish = phish.shuffle(seed=42).select(range(min_count))
balanced = concatenate_datasets([benign, phish]).shuffle(seed=42)

print()
print(f"=== Final balanced dataset ===")
print(f"Total: {len(balanced)} samples")
print(f"Labels: {pd.Series(balanced['label']).value_counts().to_string()}")

In [ ]:
train_test = balanced.train_test_split(test_size=0.2, seed=42)
train_ds = train_test["train"]
test_val = train_test["test"].train_test_split(test_size=0.5, seed=42)
val_ds = test_val["train"]
test_ds = test_val["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
for name, ds in [("Train", train_ds), ("Val", val_ds), ("Test", test_ds)]:
    counts = pd.Series(ds["label"]).value_counts()
    print(f"  {name}: benign={counts.get('benign', 0)}, phish={counts.get('phish', 0)}")

In [ ]:
max_seq_length = 512
model_id = "unsloth/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    load_in_8bit=False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=max_seq_length,
)

In [ ]:
prompt_template = """Analyze the following URL and determine if it belongs to a phishing page or a benign (legitimate) page.

URL: {url}

Classify this webpage as either "phish" or "benign". Output your prediction in XML format: <prediction>phish</prediction> or <prediction>benign</prediction>"""

def preprocess_function(examples):
    texts = []
    for url, label in zip(examples["url"], examples["label"]):
        prompt = prompt_template.format(url=url)
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": f"<prediction>{label}</prediction>"},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

In [ ]:
train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)
test_ds = test_ds.map(preprocess_function, batched=True)

print("Sample formatted text:")
print(train_ds[0]["text"][:500])

In [ ]:
train_ds[0]

In [ ]:
def evaluate_model(model, tokenizer, eval_ds, batch_size=64, max_new_tokens=20):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"

    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_template.format(url=url)}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for url in eval_ds["url"]
    ]
    labels = eval_ds["label"]

    all_responses = []
    with torch.inference_mode():
        for i in tqdm(range(0, len(prompts), batch_size)):
            inputs = tokenizer(
                prompts[i : i + batch_size], return_tensors="pt", padding=True,
            ).to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

            new_tokens = [out[len(inp):] for inp, out in zip(inputs.input_ids, outputs)]
            all_responses.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

    pred_re = re.compile(r"<prediction>(.*?)</prediction>")
    parsed = [
        m.group(1).strip().lower() if (m := pred_re.search(r)) else r.strip().lower()
        for r in all_responses
    ]

    valid_mask = [p in ("phish", "benign") for p in parsed]
    valid_labels = [l for l, v in zip(labels, valid_mask) if v]
    valid_preds = [p for p, v in zip(parsed, valid_mask) if v]

    acc = accuracy_score(labels, parsed)
    print(f"Accuracy:  {100 * acc:.2f}%")

    if valid_preds:
        print(f"Precision: {100 * precision_score(valid_labels, valid_preds, pos_label='phish'):.2f}%")
        print(f"Recall:    {100 * recall_score(valid_labels, valid_preds, pos_label='phish'):.2f}%")
        print(f"F1:        {100 * f1_score(valid_labels, valid_preds, pos_label='phish'):.2f}%")
        cm = confusion_matrix(valid_labels, valid_preds, labels=["benign", "phish"])
        print(f"\nConfusion matrix (rows=true, cols=pred):")
        print(pd.DataFrame(cm, index=["true_benign", "true_phish"], columns=["pred_benign", "pred_phish"]))

    invalid_count = len(parsed) - len(valid_preds)
    if invalid_count:
        invalid_examples = [r for r, v in zip(all_responses, valid_mask) if not v][:5]
        print(f"\nInvalid responses: {invalid_count}/{len(all_responses)}")
        print(f"Examples: {invalid_examples}")

    return {"responses": all_responses, "parsed": parsed, "labels": labels, "accuracy": acc}

In [ ]:
results = evaluate_model(model, tokenizer, val_ds)

In [ ]:
training_args = SFTConfig(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=25,
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="no",
    logging_dir="./logs",
    max_seq_length=max_seq_length,
    packing=True,
    seed=42,
    data_seed=42,
    report_to="none",
    optim="adamw_8bit",
    learning_rate=3e-04,
    lr_scheduler_type="constant_with_warmup",
    max_grad_norm=0.8,
    warmup_steps=5,
    weight_decay=0.1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    max_steps=100,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

In [ ]:
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    formatting_func=lambda examples: examples["text"],
)
trainer.can_return_loss = True

In [ ]:
trainer.train()

In [ ]:
results = evaluate_model(model, tokenizer, val_ds)

In [ ]:
results = evaluate_model(model, tokenizer, test_ds)